In [1]:
import os
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from dotenv.ipython import load_dotenv

In [2]:
load_dotenv(override=True)

True

In [3]:
api_key = os.getenv("OPENAI_API_KEY")

In [7]:
llm=ChatOpenAI(model="gpt-4o",temperature=0)

In [8]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 1. Charger .env
result = load_dotenv(override=True)
print(f"load_dotenv a retourné: {result}")

# 2. Vérifier le contenu de os.environ après chargement
print("\n=== Variables dans os.environ après load_dotenv ===")
for key in os.environ.keys():
    if 'OPENAI' in key or 'API' in key:
        print(f"{key}: {os.environ[key][:20]}...")  # Afficher les 20 premiers caractères

# 3. Récupérer la clé
api_key = os.environ.get("OPENAI_API_KEY")
print(f"\nClé API récupérée: {'Oui' if api_key else 'Non'}")
if api_key:
    print(f"Début de la clé: {api_key[:15]}...")

# 4. Initialiser avec la clé
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        api_key=api_key,  # Passer explicitement
        temperature=0.7
    )
    print("\n✅ Modèle créé avec succès!")
    
    # Tester rapidement
    response = llm.invoke("Dis 'Bonjour'")
    print(f"Test réussi: {response.content}")
    
except Exception as e:
    print(f"\n❌ Erreur: {e}")

load_dotenv a retourné: True

=== Variables dans os.environ après load_dotenv ===
OPENAI_API_KEY: sk-proj-BEah3AEZ2kpu...

Clé API récupérée: Oui
Début de la clé: sk-proj-BEah3AE...

✅ Modèle créé avec succès!

❌ Erreur: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************pWMA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


In [6]:
agent= create_agent(
    model=llm,
    system_prompt="You are a helpfull assistant"

)

In [7]:
resp=agent.invoke(input={"messages":[
    {"role":"user","content":"je m'appelle fatima "}
]})

In [8]:
print(resp['messages'][-1].content)

Bonjour Fatima ! Comment puis-je vous aider aujourd'hui ?


In [9]:
from langchain.agents.middleware import ModelRequest,ModelResponse ,wrap_model_call
from langchain.messages import HumanMessage,SystemMessage,AIMessage

In [10]:
basic_llm = ChatOpenAI(model="gpt-4o-mini",temperature=0)
advenced_llm = ChatOpenAI(model="gpt-4o",temperature=0)

In [11]:
@wrap_model_call
def dynamic_model_selection(request:ModelRequest,handler)->ModelResponse:
    env = request.runtime.context.get("env","test")
    if env == "test":
       model = basic_llm
       print("basic_llm selected")
    else:
        model = advenced_llm
        print("advenced_llm selected")
    return handler(request.override(model=model))


In [12]:
@wrap_model_call
def dynamic_model_selection(request:ModelRequest,handler)->ModelResponse:
    env = request.runtime.context.get("env","test")
    if env == "test":
       model = advenced_llm 
       print("advenced_llm selected")
    else:
        model = basic_llm
        print("basic_llm selected")
    return handler(request.override(model=model))

In [13]:
agent2 = create_agent(
    model=advenced_llm ,
    tools=[],
    middleware=[ dynamic_model_selection],
  debug= True
)

In [14]:
resp=agent2.invoke(
    input={"messages":[HumanMessage("C'est quoi un agent AI")]},
     context={"env":"test"}
    )

[values] {'messages': [HumanMessage(content="C'est quoi un agent AI", additional_kwargs={}, response_metadata={}, id='3d063eac-c0c8-4790-9ffa-f72875cafaee')]}
advenced_llm selected
[updates] {'model': {'messages': [AIMessage(content="Un agent AI (agent d'intelligence artificielle) est un système informatique conçu pour percevoir son environnement, prendre des décisions et agir de manière autonome ou semi-autonome afin d'atteindre des objectifs spécifiques. Les agents AI peuvent être utilisés dans une variété de contextes, allant des assistants virtuels comme Siri ou Alexa, aux systèmes de recommandation, en passant par les robots autonomes et les logiciels de trading algorithmique.\n\nLes agents AI fonctionnent généralement en suivant un cycle de perception-décision-action :\n\n1. **Perception** : L'agent collecte des informations sur son environnement à travers des capteurs ou des données d'entrée. Cela peut inclure des données visuelles, auditives, textuelles, etc.\n\n2. **Décision**

In [29]:
from IPython.display import Markdown

In [16]:
print(display(Markdown(resp['messages'][-1].content)))

Un agent AI (agent d'intelligence artificielle) est un système informatique conçu pour percevoir son environnement, prendre des décisions et agir de manière autonome ou semi-autonome afin d'atteindre des objectifs spécifiques. Les agents AI peuvent être utilisés dans une variété de contextes, allant des assistants virtuels comme Siri ou Alexa, aux systèmes de recommandation, en passant par les robots autonomes et les logiciels de trading algorithmique.

Les agents AI fonctionnent généralement en suivant un cycle de perception-décision-action :

1. **Perception** : L'agent collecte des informations sur son environnement à travers des capteurs ou des données d'entrée. Cela peut inclure des données visuelles, auditives, textuelles, etc.

2. **Décision** : L'agent traite les informations perçues pour prendre des décisions. Cela peut impliquer l'utilisation d'algorithmes d'apprentissage automatique, de logique floue, de systèmes experts, ou d'autres techniques d'IA pour évaluer les options possibles et choisir la meilleure action à entreprendre.

3. **Action** : L'agent exécute l'action choisie, qui peut être physique (comme déplacer un robot) ou virtuelle (comme envoyer un message ou effectuer une transaction).

Les agents AI peuvent être simples, avec des règles prédéfinies, ou très complexes, utilisant des techniques avancées d'apprentissage profond pour s'adapter et apprendre de nouvelles situations.

None


In [30]:
from langgraph.checkpoint.memory import InMemorySaver

In [31]:
memory = InMemorySaver()
agent = create_agent(
    model="openai:gpt-4o",
    system_prompt="you are a helpfull assistant",
    checkpointer = memory
)

In [35]:
# Fix: Add config with thread_id
new_config = {"configurable": {"thread_id": "session_456"}}

In [45]:
# Nouveau thread_id = nouvelle conversation
config = {"configurable": {"thread_id": 1}}

resp = agent.invoke(
    input={"messages": [HumanMessage("je m'appelle fati")]},
    config=config
)

In [46]:
print(resp['messages'][-1].content)

Enchanté de faire votre connaissance, Fati ! Comment puis-je vous aider aujourd'hui ?


In [48]:
resp = agent.invoke(
    input={"messages": [HumanMessage("c est quoi mon nom ")]},
    config= config
)

In [49]:
print(resp['messages'][-1].content) 

Vous m'avez dit que vous vous appelez Fati. Ai-je bien compris ?


In [50]:
from langchain.tools import tool 

In [53]:
@tool
def get_weather(city: str):
    """Get the weather of the given city"""  # Docstring indenté correctement
    print("weather tool invoked")
    return {
        "city": city,
        "temperature": 23,  # Correction de "teperature"
        "humidity": 88,      # Ajout de la clé "humidity" (était vide)
        "pressure": 120
    }

In [54]:
@tool
def get_employee_info(emplyee_name : str):
    """Get infos about the given employee(salary,seniority)"""  # Docstring indenté correctement
    print("get_employee_info tool invoked")
    return {
        "name": emplyee_name,
        "salary": 34000, 
        "seniority": 5    
    }

In [ ]:
agent4= create_agent(
                     model="openai:gpt-4o",
                     tools=[get_weather,get_employee_info],
                     checkpointer=memory,
                     system_prompt="answer the user question using only provided tools"
                     )

In [58]:
config = {"configurable": {"thread_id": 1}}
resp=agent4.invoke(input={'messages':[HumanMessage("La météo a casablanca")]},config=config)
print(resp['messages'][-1].content)

weather tool invoked
La météo à Casablanca indique qu'il fait 23°C avec une humidité de 88% et une pression de 120 hPa. Si vous souhaitez d'autres détails, faites-le moi savoir !


In [59]:
config = {"configurable": {"thread_id": 1}}
resp=agent4.invoke(input={'messages':[HumanMessage("quel est le salaire de hassan")]},config=config)
print(resp['messages'][-1].content)

get_employee_info tool invoked
Le salaire de Hassan est de 34 000, et il a 5 ans d'ancienneté. Si vous avez besoin de plus d'informations, n'hésitez pas à demander !


In [60]:
load_dotenv(override=True)

True

In [62]:
from langchain_tavily import TavilySearch

In [64]:
tavily = TavilySearch(max_results=10, search_depth="advanced")

ValidationError: 1 validation error for TavilySearchAPIWrapper
  Value error, Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains it, or pass `tavily_api_key` as a named parameter. [type=value_error, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

In [ ]:
@tool
def search_web(query:str):
    result = tavily.invoke({"query":query})